# **Ventes de détail : analyse du comportement des clients**

Quelle(s) stratégie(s) marketing adopter pour fidéliser les clients dans un objectif de hausse du chiffre d'affaires ?

# Importation des fichiers et nettoyage des données

## Accès depuis Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


## Importation des packages

In [ ]:
import pandas as pd
from datetime import datetime

## Lecture des fichiers

In [ ]:
chemin = '/content/gdrive/MyDrive/ProjetRetail/'

retail = pd.read_csv(chemin + 'retail_data.csv')

In [ ]:
pd.set_option('display.max_columns', None)

retail.head()

,Transaction_ID,Customer_ID,Name,Email,Phone,Address,City,State,Zipcode,Country,Age,Gender,Income,Customer_Segment,Date,Year,Month,Time,Total_Purchases,Amount,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,products
0,8691788.0,37249.0,Michelle Harrington,Ebony39@gmail.com,1.414787e+09,3959 Amanda Burgs,Dortmund,Berlin,77985.0,Germany,21.0,Male,Low,Regular,9/18/2023,2023.0,September,22:03:55,3.0,108.028757,324.086270,Clothing,Nike,Shorts,Excellent,Same-Day,Debit Card,Shipped,5.0,Cycling shorts
1,2174773.0,69749.0,Kelsey Hill,Mark36@gmail.com,6.852900e+09,82072 Dawn Centers,Nottingham,England,99071.0,UK,19.0,Female,Low,Premium,12/31/2023,2023.0,December,8:42:04,2.0,403.353907,806.707815,Electronics,Samsung,Tablet,Excellent,Standard,Credit Card,Processing,4.0,Lenovo Tab
2,6679610.0,30192.0,Scott Jensen,Shane85@gmail.com,8.362160e+09,4133 Young Canyon,Geelong,New South Wales,75929.0,Australia,48.0,Male,Low,Regular,4/26/2023,2023.0,April,4:06:29,3.0,354.477600,1063.432799,Books,Penguin Books,Children's,Average,Same-Day,Credit Card,Processing,2.0,Sports equipment
3,7232460.0,62101.0,Joseph Miller,Mary34@gmail.com,2.776752e+09,8148 Thomas Creek Suite 100,Edmonton,Ontario,88420.0,Canada,56.0,Male,High,Premium,05-08-23,2023.0,May,14:55:17,7.0,352.407717,2466.854021,Home Decor,Home Depot,Tools,Excellent,Standard,PayPal,Processing,4.0,Utility knife
4,4983775.0,27901.0,Debra Coleman,Charles30@gmail.com,9.098268e+09,5813 Lori Ports Suite 269,Bristol,England,48704.0,UK,22.0,Male,Low,Premium,01-10-24,2024.0,January,16:54:07,2.0,124.276524,248.553049,Grocery,Nestle,Chocolate,Bad,Standard,Cash,Shipped,1.0,Chocolate cookies


## Sélection des colonnes utiles et tri par date

In [ ]:
# La ville et le pays sont suffisants pour la position géographique
# Les colonnes de mois et année sont parfois inexactes
# L'heure exacte et la quantité ne seront pas explicitées
# La catégorie, la marque et le type de produit sont suffisants

retail = retail.drop(['Name','Email','Phone','Address','State','Zipcode','Year','Month','Time','Total_Purchases','products'], axis = 1)

In [ ]:
# Renommer certaines colonnes pour plus de clarté

retail = retail.rename(columns = {'Amount' : 'Unit_Price', 'Total_Amount' : 'Total_Price'})

In [ ]:
# Passage de la colonne Date au bon type, puis tri par Date
# format = 'mixed' pour tenir compte des - et /

retail.Date = pd.to_datetime(retail.Date, format = 'mixed')

retail = retail.sort_values(by = 'Date')

## NA, doublons et types



In [ ]:
# On supprime les lignes contenant des valeurs manquantes (environ 1,6%)

retail = retail.dropna(axis = 0, how = 'any')

In [ ]:
# Les Transaction_ID ne sont pas uniques, donc on supprime les doublons (environ 2,4%) de façon à ce qu'ils le soient

retail = retail.drop_duplicates(subset = ['Transaction_ID']).reset_index(drop = True)

# Vérification des doublons

print("Le dataframe contient", retail.duplicated().sum(), "doublons.")

Le dataframe contient 0 doublons.


In [ ]:
# Conversion en type int

retail.Transaction_ID = retail.Transaction_ID.astype('int')
retail.Customer_ID = retail.Customer_ID.astype('int')
retail.Age = retail.Age.astype('int')
retail.Ratings = retail.Ratings.astype('int')

# Arrondi des colonnes de prix

retail.Unit_Price = round(retail.Unit_Price).astype('int')
retail.Total_Price = round(retail.Total_Price).astype('int')

In [ ]:
retail.head()

,Transaction_ID,Customer_ID,City,Country,Age,Gender,Income,Customer_Segment,Date,Unit_Price,Total_Price,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings
0,3405226,82609,Hamburg,Germany,62,Male,Medium,New,2023-03-01,258,2575,Grocery,Nestle,Chocolate,Excellent,Express,PayPal,Processing,5
1,8374173,50529,Glasgow,UK,69,Male,Low,Regular,2023-03-01,72,144,Electronics,Sony,Smartphone,Excellent,Express,Credit Card,Delivered,5
2,5380258,15828,Chicago,USA,54,Male,Low,Regular,2023-03-01,145,872,Clothing,Adidas,Shoes,Bad,Same-Day,PayPal,Processing,1
3,6131150,49472,Oxford,UK,48,Male,High,Regular,2023-03-01,160,962,Grocery,Nestle,Chocolate,Excellent,Standard,Cash,Delivered,4
4,9360268,98463,Fort Worth,USA,19,Female,High,New,2023-03-01,26,131,Electronics,Samsung,Television,Good,Standard,PayPal,Delivered,4


# Correction et évaluation

## Ville, pays, âge, sexe, revenu et segmentation

In [ ]:
# Pour un même client, la ville, le pays, l'âge, le sexe, le revenu et la segmentation sont différents
# On corrige ce problème ici

# Création des nouvelles colonnes vides (valeur 0 arbitraire)

City = [0] * retail.shape[0]
Country = [0] * retail.shape[0]
Age = [0] * retail.shape[0]
Gender = [0] * retail.shape[0]
Income = [0] * retail.shape[0]
Segment = [0] * retail.shape[0]

# Pour chaque client, on récupère les valeurs lors du premier achat

for id in retail.Customer_ID.unique():

  city = retail[retail.Customer_ID == id].City.iloc[0]
  country = retail[retail.Customer_ID == id].Country.iloc[0]
  age = retail[retail.Customer_ID == id].Age.iloc[0]
  gender = retail[retail.Customer_ID == id].Gender.iloc[0]
  income = retail[retail.Customer_ID == id].Income.iloc[0]
  segment = retail[retail.Customer_ID == id].Customer_Segment.iloc[0]

# On récupère les index des transactions de ce client, et on met à jour les nouvelles colonnes aux positions correspondantes

  for transaction in retail[retail.Customer_ID == id].index:

    City[transaction] = city
    Country[transaction] = country
    Age[transaction] = age
    Gender[transaction] = gender
    Income[transaction] = income
    Segment[transaction] = segment

# Affectation des nouvelles colonnes

retail['City'] = City
retail['Country'] = Country
retail['Age'] = Age
retail['Gender'] = Gender
retail['Income'] = Income
retail['Customer_Segment'] = Segment

## Segmentation des clients

In [ ]:
# Pour évaluer l'utilisabilité de la colonne Customer_Segment, on va comparer ses valeurs en créant soi-même
# Une segmentation ayant approximativement le même nombre de clients dans chaque catégorie

# On commence par une colonne vide

Seg = [0] * retail.shape[0]

# Pour chaque client, on récupère la date de dernier achat, le nombre d'achats et le montant total

for id in retail.Customer_ID.unique():

  last_purchase = retail[retail.Customer_ID == id].Date.iloc[-1]
  num_purchases = len(retail[retail.Customer_ID == id])
  amount_spent = round(sum(retail[retail.Customer_ID == id].Total_Price), 2)

# Selon les quintiles, on attribue un score de récence, fréquence et montant

  if last_purchase < datetime(2023, 5, 13):
    R = 1
  elif last_purchase < datetime(2023, 7, 25):
    R = 2
  elif last_purchase < datetime(2023, 10, 6):
    R = 3
  elif last_purchase < datetime(2023, 12, 18):
    R = 4
  else:
    R = 5

  if num_purchases < 3:
    F = 1
  elif num_purchases < 4:
    F = 2
  elif num_purchases < 5:
    F = 3
  elif num_purchases < 6:
    F = 4
  else:
    F = 5

  if amount_spent < 1850:
    M = 1
  elif amount_spent < 3380:
    M = 2
  elif amount_spent < 4935:
    M = 3
  elif amount_spent < 7120:
    M = 4
  else:
    M = 5

# 3-9 : Inactif
# 10-13 : Régulier
# 14-15 : VIP

  if R + F + M <= 9:
    for transaction in retail[retail.Customer_ID == id].index:
      Seg[transaction] = 'New'
  elif R + F + M <= 13:
    for transaction in retail[retail.Customer_ID == id].index:
      Seg[transaction] = 'Regular'
  else:
    for transaction in retail[retail.Customer_ID == id].index:
      Seg[transaction] = 'Premium'

retail['Segment'] = Seg

In [ ]:
# Comparaison

sim = sum(retail.Customer_Segment == retail.Segment)

print("Le taux de similitude entre Customer_Segment et Segment est", round(100 * sim / retail.shape[0], 1), "%.")

Le taux de similitude entre Customer_Segment et Segment est 36.0 %.


In [ ]:
# Le taux de similitude étant très bas, on ne va donc pas pouvoir s'en servir tel quel

retail = retail.drop(['Customer_Segment','Segment'], axis = 1)

# Segmentation RFM

In [ ]:
# Nouveau dataframe pour la segmentation des clients

retail_RFM = pd.DataFrame(retail.Customer_ID).drop_duplicates()

In [ ]:
# On crée proprement une colonne de segmentation RFM avec les types de profil possibles

Last, Number, Amount = [], [], []

RFM = []

for id in retail_RFM.Customer_ID:

  last_purchase = retail[retail.Customer_ID == id].Date.iloc[-1]
  num_purchases = len(retail[retail.Customer_ID == id])
  amount_spent = round(sum(retail[retail.Customer_ID == id].Total_Price), 2)

  if last_purchase < datetime(2023, 5, 13):
    R = 1
  elif last_purchase < datetime(2023, 7, 25):
    R = 2
  elif last_purchase < datetime(2023, 10, 6):
    R = 3
  elif last_purchase < datetime(2023, 12, 18):
    R = 4
  else:
    R = 5

  if num_purchases < 3:
    F = 1
  elif num_purchases < 4:
    F = 2
  elif num_purchases < 5:
    F = 3
  elif num_purchases < 6:
    F = 4
  else:
    F = 5

  if amount_spent < 1850:
    M = 1
  elif amount_spent < 3380:
    M = 2
  elif amount_spent < 4935:
    M = 3
  elif amount_spent < 7120:
    M = 4
  else:
    M = 5

# Pour réduire la dimension, on multiplie F et M entre eux

  if R in [1,2]:
    if F*M <= 5:
      Profil = 'Lost'
    elif F*M <= 15:
      Profil = 'At risk'
    else:
      Profil = 'Cannot lose them'
  elif R == 3:
    if F*M <= 5:
      Profil = 'About to sleep'
    elif F*M <= 9:
      Profil = 'Need attention'
    else :
      Profil = 'Loyal'
  elif R == 4:
    if F*M <= 3:
      Profil = 'Promising'
    elif F*M <= 9:
      Profil = 'Potential loyalists'
    else:
      Profil = 'Loyal'
  else:
    if F*M <= 3:
      Profil = 'New'
    elif F*M <= 9:
      Profil = 'Potential loyalists'
    else:
      Profil = 'Champions'

  Last.append(last_purchase)
  Number.append(num_purchases)
  Amount.append(amount_spent)
  RFM.append(Profil)

# Création des nouvelles colonnes

retail_RFM['Last_purchase'] = Last
retail_RFM['Number_of_purchases'] = Number
retail_RFM['Total_amount_spent'] = Amount
retail_RFM['Category'] = RFM

In [ ]:
# Remplacement de l'index par la colonne Customer_ID

retail_RFM = retail_RFM.set_index('Customer_ID')

retail_RFM.head()

,Last_purchase,Number_of_purchases,Total_amount_spent,Category
Customer_ID,,,,
82609,2023-11-01,2,2841,Promising
50529,2023-09-26,6,9268,Loyal
15828,2023-07-21,3,3423,At risk
49472,2023-04-24,2,1484,Lost
98463,2023-12-28,9,15614,Champions


# Sauvegarde pour Power BI

In [ ]:
# Sauvegarde au format CSV

retail.to_csv("/content/gdrive/MyDrive/ProjetRetail/PowerBI_Retail_data.csv", index = False)

retail_RFM.to_csv("/content/gdrive/MyDrive/ProjetRetail/PowerBI_Retail_RFM_data.csv")